# Understanding the AST dependency graph

This notebook explains, runnably, the machinery behind
**`claude_code_from_scratch_v3_GRAPH.html`** — the interactive map where every notebook
*code cell* is a node and every *edge* is a real dependency between cells.

The whole graph rests on one idea: **you can read a block of Python without running it** by
parsing it into an **Abstract Syntax Tree (AST)** and asking two questions of every name in it:

- which names does this block **define**? (functions, classes, assignments)
- which names does this block **use**? (loads / references)

If cell *A* defines `llm` and cell *B* uses `llm`, then `B` depends on `A` — that's an edge
`A → B`. Nothing about this needs the model, the network, or even running the code. It is pure
*static analysis* of the source text.

We build that up in five steps:

1. what the parser sees (`ast.dump`)
2. the same thing drawn as a tree
3. the key trick: `Store` vs `Load` = *defines* vs *uses*
4. from two blocks to an edge
5. the real thing — run it on the actual v3 notebook, with a `show_ast(cell)` helper

> Everything here is standard-library `ast`. No third-party packages, no Ollama, no network.

## 0. A small specimen

We'll dissect this one short function for most of the notebook.

In [1]:
import ast, textwrap

SNIPPET = textwrap.dedent("""
    def write_code(filename, content):
        res = lint_python(content)
        path = AGENT_CODE_DIR / filename
        return f"WROTE {filename}"
""").strip()

print(SNIPPET)

def write_code(filename, content):
    res = lint_python(content)
    path = AGENT_CODE_DIR / filename
    return f"WROTE {filename}"


## 1. What the parser sees — `ast.dump`

`ast.parse(source)` turns text into a tree of **nodes**, each one a grammar construct:
`Module`, `FunctionDef`, `Assign`, `Call`, `Name`, `Constant`, and so on. `ast.dump` prints
that tree as text. It's verbose, but every field matters — notice especially the `ctx=Store()`
vs `ctx=Load()` markers on each `Name`, which we'll exploit in step 3.

In [2]:
tree = ast.parse(SNIPPET)
print(ast.dump(tree, indent=4))

Module(
    body=[
        FunctionDef(
            name='write_code',
            args=arguments(
                args=[
                    arg(arg='filename'),
                    arg(arg='content')]),
            body=[
                Assign(
                    targets=[
                        Name(id='res', ctx=Store())],
                    value=Call(
                        func=Name(id='lint_python', ctx=Load()),
                        args=[
                            Name(id='content', ctx=Load())])),
                Assign(
                    targets=[
                        Name(id='path', ctx=Store())],
                    value=BinOp(
                        left=Name(id='AGENT_CODE_DIR', ctx=Load()),
                        op=Div(),
                        right=Name(id='filename', ctx=Load()))),
                Return(
                    value=JoinedStr(
                        values=[
                            Constant(value='WROTE '),
                    

## 2. The same tree, drawn as a tree

`ast.dump` is the *textual* form. The tree structure is easier to see if we walk it with
`ast.iter_child_nodes` (each node's direct children) and print with indentation. This is exactly
the structure the graph generator traverses.

In [3]:
def draw(node, prefix="", last=True):
    """Pretty-print an AST as an indented tree, annotating the fields that matter."""
    label = type(node).__name__
    extra = ""
    if isinstance(node, ast.Name):          extra = f"  id={node.id!r} ({type(node.ctx).__name__})"
    elif isinstance(node, ast.FunctionDef):  extra = f"  name={node.name!r}"
    elif isinstance(node, ast.ClassDef):     extra = f"  name={node.name!r}"
    elif isinstance(node, ast.arg):          extra = f"  arg={node.arg!r}"
    elif isinstance(node, ast.Constant):     extra = f"  value={node.value!r}"
    elif isinstance(node, ast.Attribute):    extra = f"  attr={node.attr!r}"
    print(prefix + ("└─ " if last else "├─ ") + label + extra)
    kids = list(ast.iter_child_nodes(node))
    child_prefix = prefix + ("   " if last else "│  ")
    for i, ch in enumerate(kids):
        draw(ch, child_prefix, i == len(kids) - 1)

draw(tree)

└─ Module
   └─ FunctionDef  name='write_code'
      ├─ arguments
      │  ├─ arg  arg='filename'
      │  └─ arg  arg='content'
      ├─ Assign
      │  ├─ Name  id='res' (Store)
      │  │  └─ Store
      │  └─ Call
      │     ├─ Name  id='lint_python' (Load)
      │     │  └─ Load
      │     └─ Name  id='content' (Load)
      │        └─ Load
      ├─ Assign
      │  ├─ Name  id='path' (Store)
      │  │  └─ Store
      │  └─ BinOp
      │     ├─ Name  id='AGENT_CODE_DIR' (Load)
      │     │  └─ Load
      │     ├─ Div
      │     └─ Name  id='filename' (Load)
      │        └─ Load
      └─ Return
         └─ JoinedStr
            ├─ Constant  value='WROTE '
            └─ FormattedValue
               └─ Name  id='filename' (Load)
                  └─ Load


Read it top-down:

- `Module` is always the root; its `body` is the list of top-level statements.
- `FunctionDef name='write_code'` — the `def`. Its name is a thing this block **creates**.
- inside, two `Assign` nodes create `res` and `path` — note their `Name` has `ctx=Store`.
- the calls/expressions reference `lint_python`, `AGENT_CODE_DIR`, `content`, `filename` —
  those `Name`s have `ctx=Load`.

That `Store` / `Load` distinction is the whole game.

## 3. `Store` vs `Load` = *defines* vs *uses*

Python's compiler tags every `Name` with a **context**:

- **`Store`** — the name is being *bound* (left of `=`, a `for` target, …). Plus `FunctionDef`
  and `ClassDef` *names* are definitions too. → these are what the block **defines**.
- **`Load`** — the name is being *read*. → these are what the block **uses**.

`ast.walk(tree)` yields every node in the tree (the flattened version of `draw`), so we can
collect both sets in a single pass. This is the heart of the generator's `parse_cell`.

In [4]:
def parse_cell(src):
    """Return (defines, uses) for one block of source — module-level definitions, all loads."""
    defines, uses = set(), set()
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return defines, uses
    # module-level definitions: def / class / top-level assignment targets
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            defines.add(n.name)
        elif isinstance(n, ast.Assign):
            for t in n.targets:
                if isinstance(t, ast.Name):
                    defines.add(t.id)
    # uses: every Name read anywhere in the tree
    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load):
            uses.add(node.id)
    return defines, uses

d, u = parse_cell(SNIPPET)
print("defines:", sorted(d))
print("uses   :", sorted(u))

defines: ['write_code']
uses   : ['AGENT_CODE_DIR', 'content', 'filename', 'lint_python']


Notice `filename` and `content` show up in **uses** even though they're the function's
parameters. That's fine: they aren't *module-level* definitions, so no other cell can possibly
define them — when we resolve uses against the define-map in the next step, they simply match
nothing and produce no edge. The graph stays clean without us special-casing locals.

(`parse_cell` deliberately looks only at **module-level** `def`/`class`/assignment for *defines*
— those are the names a cell exports to the rest of the notebook. It scans **all** `Load` names
for *uses*, because a dependency can hide anywhere inside a function body.)

## 4. From two blocks to an edge

Now the actual graph logic, in miniature. Given several blocks:

1. build a map `symbol → the block that defines it`;
2. for each block, every *use* that resolves to a **different** block's *define* becomes an edge.

Here are two toy cells where the dependency is obvious.

In [5]:
CELL_A = textwrap.dedent("""
    AGENT_CODE_DIR = '/tmp/agent_code'
    def lint_python(code):
        return {'passed': True}
""").strip()

CELL_B = SNIPPET   # uses lint_python AND AGENT_CODE_DIR, both defined in CELL_A

blocks = {0: CELL_A, 1: CELL_B}

# 1) symbol -> defining block (first definer wins)
defmap = {}
parsed = {}
for idx, src in blocks.items():
    defs, uses = parse_cell(src)
    parsed[idx] = (defs, uses)
    for name in defs:
        defmap.setdefault(name, idx)

# 2) edges: a use that resolves to another block
edges = []
for idx, (defs, uses) in parsed.items():
    by_src = {}
    for name in uses:
        src_idx = defmap.get(name)
        if src_idx is not None and src_idx != idx:
            by_src.setdefault(src_idx, []).append(name)
    for src_idx, syms in by_src.items():
        edges.append((src_idx, idx, sorted(set(syms))))

print("symbol -> defining block:", defmap)
print("\nedges (from -> to  via symbols):")
for a, b, syms in edges:
    print(f"  #{a} -> #{b}   {syms}")

symbol -> defining block: {'lint_python': 0, 'AGENT_CODE_DIR': 0, 'write_code': 1}

edges (from -> to  via symbols):
  #0 -> #1   ['AGENT_CODE_DIR', 'lint_python']


That single edge `#0 → #1 via ['AGENT_CODE_DIR', 'lint_python']` is exactly the kind of arrow
you see in the HTML graph, label and all. Scale this loop from 2 blocks to the notebook's 39
code cells and you have the whole picture — no heuristics, no model, just `Store`/`Load`
bookkeeping.

## 5. The real thing — run it on the v3 notebook

Now we point the same `parse_cell` at the actual `claude_code_from_scratch_v3.ipynb`: assign
each code cell to its `## Phase` heading, compute the define-map and the edges, and print the
graph's shape. If the notebook isn't found next to this one, this section just skips.

In [6]:
import json, re
from pathlib import Path

NB = Path.cwd() / "claude_code_from_scratch_v3.ipynb"
have_nb = NB.exists()
print("notebook found:" , have_nb, "->", NB if have_nb else "(skipping section 5)")

cells_v3 = []   # list of (idx, phase, src)
if have_nb:
    nb = json.loads(NB.read_text())
    phase = "Phase 0"
    for i, c in enumerate(nb["cells"]):
        src = "".join(c["source"])
        if c["cell_type"] == "markdown":
            for line in src.splitlines():
                if line.startswith("## "):
                    phase = line[3:].strip(); break
        elif src.strip():
            cells_v3.append((i, phase, src))
    print(f"{len(cells_v3)} code cells")

notebook found: True -> /home/bmartins/dev/agentic_patterns/src/code_assistant/claude_code_from_scratch_v3.ipynb
39 code cells


In [7]:
if have_nb:
    # define-map across the real notebook
    defmap_v3, parsed_v3 = {}, {}
    for idx, ph, src in cells_v3:
        defs, uses = parse_cell(src)
        parsed_v3[idx] = (defs, uses)
        for name in defs:
            defmap_v3.setdefault(name, idx)

    edges_v3 = []
    for idx, ph, src in cells_v3:
        defs, uses = parsed_v3[idx]
        deps = sorted({defmap_v3[n] for n in uses if n in defmap_v3 and defmap_v3[n] != idx})
        for src_idx in deps:
            edges_v3.append((src_idx, idx))

    print(f"nodes (code cells): {len(cells_v3)}")
    print(f"edges (dependencies): {len(edges_v3)}")
    # the most depended-upon cells = the notebook's hubs
    from collections import Counter
    indeg_targets = Counter(a for a, _ in edges_v3)   # how often each cell is a *source* (used by others)
    print("\nmost-used cells (a source of many edges):")
    for idx, n in indeg_targets.most_common(5):
        title = next(p for i, p, s in cells_v3 if i == idx)
        print(f"  cell #{idx:2d} used by {n:2d} others   [{title[:34]}]")

nodes (code cells): 39
edges (dependencies): 128

most-used cells (a source of many edges):
  cell # 3 used by 21 others   [Phase 0 — Setup: imports, logging,]
  cell # 7 used by 17 others   [Phase 0.5 — Observability: a callb]
  cell # 4 used by 10 others   [Phase 0 — Setup: imports, logging,]
  cell # 5 used by 10 others   [Phase 0 — Setup: imports, logging,]
  cell #10 used by 10 others   [Phase 1 — Cognitive substrate: thi]


The hubs that surface — typically the `llm()` factory cell and the tracer cell — are exactly the
ones the HTML graph shows everything pointing back at. The census matches the numbers baked into
the graph (39 nodes, ~128 edges).

### A `show_ast(cell)` helper

Finally, the on-demand inspector. Give it a cell index from the v3 notebook and it prints that
cell's AST tree plus its computed *defines / depends-on / used-by* — the same three facts the
HTML side panel shows when you click a node.

In [8]:
def show_ast(cell_idx, tree_depth=True):
    """Inspect one v3 code cell: its AST tree + defines / depends-on / used-by."""
    if not have_nb:
        print("v3 notebook not available."); return
    match = [(i, p, s) for i, p, s in cells_v3 if i == cell_idx]
    if not match:
        print(f"cell #{cell_idx} is not a code cell. Try one of:",
              [i for i, _, _ in cells_v3][:20], "..."); return
    idx, phase, src = match[0]
    defs, uses = parsed_v3[idx]
    depends_on = {}
    for n in uses:
        s = defmap_v3.get(n)
        if s is not None and s != idx:
            depends_on.setdefault(s, []).append(n)
    used_by = {}
    for j, (jd, ju) in parsed_v3.items():
        if j == idx: continue
        shared = [n for n in ju if n in defs]
        if shared:
            used_by[j] = sorted(set(shared))

    print(f"=== cell #{idx}  [{phase}] ===\n")
    print("SOURCE:\n" + src + "\n")
    if tree_depth:
        print("AST TREE:")
        draw(ast.parse(src))
        print()
    print("defines    :", sorted(defs))
    print("depends on :", {f"#{k}": v for k, v in sorted(depends_on.items())})
    print("used by    :", {f"#{k}": v for k, v in sorted(used_by.items())})

# Example: cell 16 defines write_code (and the tool registry); it depends on the config cell.
show_ast(16)

=== cell #16  [Phase 2 — Tools: `@tool`-decorated, sandboxed to the workspace] ===

SOURCE:
"""Coding-specific tools: lint-gated write, python runner, test runner (v2 bodies, @tool-wrapped)."""


def lint_python(code: str) -> dict:
    """Lightweight static gate: must compile; flag bare excepts. (Plain fn -- reused internally.)"""
    errors = []
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as tmp:
        tmp.write(code); tmp_path = tmp.name
    try:
        py_compile.compile(tmp_path, doraise=True)
    except py_compile.PyCompileError as e:
        errors.append(f"SyntaxError: {e}")
    finally:
        os.unlink(tmp_path)
    try:
        for node in ast.walk(ast.parse(code)):
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                errors.append(f"Style: bare `except:` at line {node.lineno}")
    except SyntaxError:
        pass
    return {"passed": len(errors) == 0, "errors": errors}


def _run_tests(test_code: str, 

Try `show_ast(5)` (the `llm()` factory — a big hub), `show_ast(18)` (the tool-loop graph), or
`show_ast(37)` (the five-subagent team — depends on the most other cells). Pass
`tree_depth=False` to skip the full tree and just see the defines/depends/used-by triple.

## Recap — how this maps to the artifacts

| concept here | in the HTML graph | in the prose `.md` |
|---|---|---|
| a parsed code cell | a **node** (shows the cell's code) | a "cell N" reference |
| `parse_cell` *defines* | the node's **Defines** list | the functions documented in that section |
| `parse_cell` *uses* → resolved | an **edge** `A → B` (labelled with symbols) | "depends on / used by" |
| `show_ast(cell)` | clicking a node → the **side panel** | the per-cell write-up |

The generator that does this for real and emits the HTML is **`_build_v3_graph.py`** (run
`python3 _build_v3_graph.py` to rebuild `claude_code_from_scratch_v3_GRAPH.html`). The only
substantive differences from this notebook are cosmetic: it also carries a human-written title
and one-line description per cell, and renders everything as draggable boxes with SVG edges.

**The takeaway:** a dependency graph over code is not magic or model-driven — it's `ast.parse`
plus the `Store`/`Load` distinction, counted across blocks.